In [1]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_core.documents import Document
import os
import yaml
from pathlib import Path
import chromadb
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction
import shutil
import hashlib

# Пути и параметры
md_path = Path('c:/Users/Alkor/gd/news_rss_md_rts_21-00')
chromadb_path = './chroma_db_ollama_graph_rts_21-00'
model_name = "bge-m3"
url_ai = "http://localhost:11434/api/embeddings"

def get_folder_size(folder_path):
    total_size = 0
    for dirpath, _, filenames in os.walk(folder_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)  # Размер в МБ

def load_markdown_files(directory):
    documents = []
    for file_path in list(directory.glob("**/*.md")):
        # Чтение содержимого файла
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        
        # Разделение метаданных и текста
        if content.startswith('---'):
            parts = content.split('---', 2)
            if len(parts) >= 3:
                metadata_yaml = parts[1].strip()
                text_content = parts[2].strip()
                # Парсинг метаданных
                metadata = yaml.safe_load(metadata_yaml) or {}
                # Преобразование метаданных в строки
                metadata_str = {
                    "next_bar": str(metadata.get("next_bar", "unknown")),
                    "date_min": str(metadata.get("date_min", "unknown")),
                    "date_max": str(metadata.get("date_max", "unknown")),
                    "source": file_path.name,
                    "date": file_path.stem
                }
                # Создание объекта Document
                doc = Document(
                    page_content=text_content,
                    metadata=metadata_str
                )
                documents.append(doc)
            else:
                # Если нет метаданных, добавляем unknown
                doc = Document(
                    page_content=content,
                    metadata={
                        "next_bar": "unknown",
                        "date_min": "unknown",
                        "date_max": "unknown",
                        "source": file_path.name,
                        "date": file_path.stem
                    }
                )
                documents.append(doc)
        else:
            # Если нет секции метаданных
            doc = Document(
                page_content=content,
                metadata={
                    "next_bar": "unknown",
                    "date_min": "unknown",
                    "date_max": "unknown",
                    "source": file_path.name,
                    "date": file_path.stem
                }
            )
            documents.append(doc)
    return documents

# Удаление папки chroma_db, если она существует
if os.path.exists(chromadb_path):
    print(f"Размер папки {chromadb_path} до удаления: {get_folder_size(chromadb_path):.2f} МБ")
    shutil.rmtree(chromadb_path)
    print(f"Папка {chromadb_path} удалена.")

# Инициализация клиента ChromaDB
client = chromadb.PersistentClient(path=chromadb_path)

# Создание функции эмбеддингов для Ollama
ef = OllamaEmbeddingFunction(
    model_name=model_name,
    url=url_ai
)

# Создание коллекции
collection = client.create_collection(name="news_collection", embedding_function=ef)

# Загрузка Markdown-файлов
documents = load_markdown_files(md_path)

# Проверка на пустую папку
if not documents:
    print("Не найдено Markdown-файлов в указанной директории.")
    exit(1)
else:
    print(f"Загружено {len(documents)} Markdown-файлов из {md_path}")
    print(f"Документы даты: {set(doc.metadata['date'] for doc in documents)}")
    print(f"Направление следующего бара: {set(doc.metadata['next_bar'] for doc in documents)}")
    print(f"Минимальные даты: {set(doc.metadata['date_min'] for doc in documents)}")
    print(f"Максимальные даты: {set(doc.metadata['date_max'] for doc in documents)}")

# Подготовка данных для ChromaDB
doc_texts = [doc.page_content for doc in documents]
doc_ids = [hashlib.md5(doc.page_content.encode()).hexdigest() for doc in documents]
doc_metadatas = [doc.metadata for doc in documents]

# Добавление в коллекцию
collection.add(
    ids=doc_ids,
    documents=doc_texts,
    metadatas=doc_metadatas
)

# # Пример поиска с фильтрацией по метаданным
# query = "Новости о Tesla"
# results = collection.query(
#     query_texts=[query],
#     n_results=3,
#     where={"next_bar": "up"}  # Фильтрация по next_bar
#     # where={"next_bar": "up", "date_min": {"$eq": "2025-07-27 21:00:00"}}
# )
# print(results)

Размер папки ./chroma_db_ollama_graph_rts_21-00 до удаления: 52.47 МБ
Папка ./chroma_db_ollama_graph_rts_21-00 удалена.
Загружено 23 Markdown-файлов из c:\Users\Alkor\gd\news_rss_md_rts_21-00
Документы даты: {'2025-06-27', '2025-06-30', 'current', '2025-06-25', '2025-07-09', '2025-07-01', '2025-07-02', '2025-07-22', '2025-07-10', '2025-07-16', '2025-07-17', '2025-07-07', '2025-07-18', '2025-07-21', '2025-07-04', '2025-07-24', '2025-07-03', '2025-06-26', '2025-07-08', '2025-07-11', '2025-07-15', '2025-07-23', '2025-07-14'}
Направление следующего бара: {'down', 'up', 'current'}
Минимальные даты: {'2025-07-14 18:00:00', '2025-07-21 18:00:00', '2025-07-10 18:00:00', '2025-07-04 18:00:00', '2025-07-16 18:00:00', '2025-06-24 18:00:00', '2025-07-08 18:00:00', '2025-07-01 18:00:00', '2025-07-22 18:00:00', '2025-07-11 18:00:00', '2025-07-02 18:00:00', '2025-06-25 18:00:00', '2025-07-17 18:00:00', '2025-07-09 18:00:00', '2025-07-03 18:00:00', '2025-07-07 18:00:00', '2025-06-27 18:00:00', '2025-0

In [2]:
# from sklearn.manifold import TSNE
# import numpy as np
# import plotly.graph_objects as go

# # Теперь мы можем визуализировать векторы с помощью t-SNE
# # t-SNE - это метод, который позволяет визуализировать высокоразмерные данные
# result = collection.get(include=['embeddings', 'documents', 'metadatas'])
# vectors = np.array(result['embeddings'])
# documents = result['documents']
# metadatas = result['metadatas']
# doc_types = [metadata['next_bar'] for metadata in metadatas]
# colors = [['blue', 'red', 'black'][['up', 'down', 'current'].index(t)] for t in doc_types]


# # Нам, людям, проще визуализировать объекты в 2D!
# # Уменьшите размерность векторов до 2D, используя t-SNE
# # (t-распределенное стохастическое вложение соседей)
# tsne = TSNE(n_components=2, random_state=42, perplexity=3)
# reduced_vectors = tsne.fit_transform(vectors)

# # Create the 2D scatter plot
# fig = go.Figure(data=[go.Scatter(
#     x=reduced_vectors[:, 0],
#     y=reduced_vectors[:, 1],
#     mode='markers',
#     marker=dict(size=5, color=colors, opacity=0.8),
#     text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
#     hoverinfo='text'
# )])

# fig.update_layout(
#     title=f'2D Chroma Vector Store Ollama (bge-m3) 2025-07-24',
#     xaxis_title='x',
#     yaxis_title='y',
#     width=800,
#     height=600,
#     margin=dict(r=20, b=10, l=10, t=40)
# )

# fig.show()

In [3]:
from sklearn.manifold import TSNE
import numpy as np
import plotly.graph_objects as go
from datetime import datetime

# Пути и параметры
output_dir = Path('C:/Users/Alkor/gd/rss_news_predict_quote/Ollama_predict_img_rts_21-00')

# Создаем директорию для сохранения изображений, если она не существует
output_dir.mkdir(parents=True, exist_ok=True)

# Теперь мы можем визуализировать векторы с помощью t-SNE
# t-SNE - это метод, который позволяет визуализировать высокоразмерные данные
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['next_bar'] for metadata in metadatas]
doc_dates = [metadata['date'] for metadata in metadatas]
doc_date_mins = [metadata['date_min'] for metadata in metadatas]
colors = [['blue', 'red', 'black'][['up', 'down', 'current'].index(t)] for t in doc_types]

# Нам, людям, проще визуализировать объекты в 2D!
# Уменьшите размерность векторов до 2D, используя t-SNE
# (t-распределенное стохастическое вложение соседей)
tsne = TSNE(n_components=2, random_state=42, perplexity=3)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Date: {dt}    Date Min: {dm}<br>Text: {d[:100]}..." 
          for t, dt, dm, d in zip(doc_types, doc_dates, doc_date_mins, documents)],
    hoverinfo='text'
)])

# Всталяем дату
# graph_date = datetime.now().strftime('%Y-%m-%d %HH:%MM:%SS')
graph_date = '2025-07-28'  # Можно заменить на динамическое значение, если нужно

fig.update_layout(
    title=f'2D Chroma Vector Store Ollama (bge-m3) {graph_date}',
    xaxis_title='x',
    yaxis_title='y',
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

# Сохранение графика в файл
output_file = output_dir / f"{graph_date}.png"
fig.write_image(output_file)

fig.show()

In [4]:
# Let's try 3D!
tsne = TSNE(n_components=3, random_state=42, perplexity=5)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()